# Bead Shock Curve Fits

Status: cleaned for the publication supplement. Supported cells start from the curated time-series data committed under `data/time-series/` and write generated files under ignored `outputs/` directories.

Upstream image/video processing, raw acquisition data, segmentation outputs, bead-center detection outputs, and other large intermediates are intentionally not included in this repository. Cells from the original notebook that depended on those local upstream files have been replaced by archived notes.


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from scipy.optimize import curve_fit
import scipy.signal
import json
import re
from scipy.interpolate import interp1d
import scipy.signal as signal
from sklearn.metrics import r2_score
from scipy.stats import linregress

import sys
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "bead_time_series.py").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR))
else:
    sys.path.insert(0, str((NOTEBOOK_DIR / "code" / "bead-assay").resolve()))
from bead_time_series import ROOT, legacy_speed_tree, legacy_condition_folder, read_legacy_speed_csv, iter_condition_traces, median_kernel_size

# Ensure plots are rendered in the notebook
%matplotlib inline

## Curve fitting one motor speed to extract magnitude and timescales of speed change

In [ ]:
# Define exponential functions
def exp_decrease(t, A, tau, C, t0):
    return A / (1 + np.exp((t-175-t0) / tau)) + C

def exp_increase(t, B, tau, D):
    return B * (1 - np.exp(-(t - 270) / tau)) + D

def calculate_time_constants(time, speed, filename):
    results = []

    #normalize speed for ease of fitting
    speed = speed / np.average(speed[time<=180]) #normally 300

    # Filter data for decrease and increase
    mask_decrease = (time >= 175) & (time <= 240)
    time_decrease = time[mask_decrease]
    speed_decrease = speed[mask_decrease]

    mask_increase = (time > 270) & (time <= 360)
    time_increase = time[mask_increase]
    speed_increase = speed[mask_increase]

    # Fit the decrease segment
    try:
        popt_decrease, _ = curve_fit(
            exp_decrease,
            time_decrease,
            speed_decrease,
            p0=[1-np.min(speed_decrease), 10, np.min(speed_decrease), 10],
        )
        A_dec, tau_dec, C_dec, t0_dec = popt_decrease
    except Exception as e:
        print(f"Error fitting decrease for {filename}: {e}")
        tau_dec = np.nan
        A_dec, tau_dec, C_dec, t0_dec = [np.nan, np.nan, np.nan, np.nan]
        popt_decrease = [np.nan, np.nan, np.nan, np.nan]

    # if True: #plotting to double check
    #     if filename == "500": #filename[-13:-10]
    #         plt.plot(time, speed)
    #         plt.plot(time_decrease, exp_decrease(time_decrease, *popt_decrease))

    # Fit the increase segment
    try:
        popt_increase, _ = curve_fit(
            exp_increase, time_increase, speed_increase, p0=[1, 50, 0.5]
        )
        B_inc, tau_inc, D_inc = popt_increase
    except Exception as e:
        print(f"Error fitting increase for {filename}: {e}")
        tau_inc = np.nan
        popt_increase = [np.nan, np.nan, np.nan]

    # Save results for this concentration
    condition = filename.split(".csv")[
        0
    ]  # Assuming the filename contains the concentration info
    results.append(
        {
            "Condition": condition,
            "Tau (Decrease)": tau_dec,
            "Tau (Increase)": tau_inc,
            "Time Range (Decrease)": f"180-270",
            "Time Range (Increase)": f"270-{time[-1]}",
        }
    )

    return tau_dec, t0_dec, A_dec, tau_inc

In [ ]:
def convert_to_300fps(time_stamps, data, target_fps=300):# Calculate the total duration
    total_time = time_stamps[-1]
    
    # Generate new time stamps at the target FPS
    new_time_stamps = np.linspace(0, total_time, int(total_time * target_fps))
    
    # Create a linear interpolator
    interpolator = interp1d(time_stamps, data, kind='linear', fill_value='extrapolate')
    
    # Interpolate data to the new time stamps
    interpolated_data = interpolator(new_time_stamps)
    
    return new_time_stamps[:380*target_fps], interpolated_data[:380*target_fps]

def bin_data(time_stamps, data, bin_size=0.1):
    data = data[time_stamps < 380]
    time_stamps = time_stamps[time_stamps < 380]
    
    # Determine the range of time stamps
    start_time = 0
    end_time = 380
    
    # Create bin edges
    bin_edges = np.arange(start_time, end_time + bin_size, bin_size)
    
    # Calculate bin indices for each time stamp
    bin_indices = np.digitize(time_stamps, bin_edges, right=False) - 1  # Bin indices (0-based)

    # Initialize lists to store results
    binned_times = np.arange(0,end_time,0.1)
    averaged_data = np.zeros_like(binned_times)
    
    # Iterate over each bin
    for i in range(len(bin_edges) - 1):
        # Get indices of data points in the current bin
        indices_in_bin = np.where(bin_indices == i)[0]
        
        if len(indices_in_bin) > 0:
            # Calculate the midpoint of the current bin
            bin_midpoint = (bin_edges[i] + bin_edges[i + 1]) / 2
            binned_times[i] = (bin_midpoint)
            
            # Calculate the average of data points in the current bin
            average_value = np.mean(data[indices_in_bin])
            averaged_data[i] = (average_value)
    
    return np.array(binned_times), np.array(averaged_data)

def bin_data_2(time_stamps, data, bin_size=0.1):
    T = time_stamps[time_stamps<=380]
    F = data[time_stamps<=380]
    
    # Resample into 4000 bins
    data_F = np.zeros(3800)
    data_T = np.zeros(3800)

    bin = 0
    counter = 0

    for kk in range(len(T)):
        counter += 1
        pos = int(T[kk] // 0.1)
        if pos < 3800:
            data_F[pos] += F[kk]
            if T[kk] // 0.1 > bin + 1:
                data_F[pos - 1] /= counter
                data_T[pos - 1] = (bin + 0.5) * 0.1
                counter = 0
                bin += 1
        else:
            data_F[pos - 1] /= counter
            data_T[pos - 1] = (bin + 0.5) * 0.1
            break
    
    return data_T, data_F

In [ ]:
def logarithmic_fit(x, A, B, C):
    return np.where(B*x<=1, 0, A * np.log(B * (x)))

def linear_fit(x, A, B):
    return A * x 

def simple_log(x, A, B):
    return np.where(x<1, 0, A * np.log(x) + B)

## Import data from CSV files to fit curves and extract timescales (tau)

### Archived upstream-dependent cell

This original code cell depended on local upstream files or intermediate products that are intentionally not included in this time-series-first supplement. The supported workflow starts from `data/time-series/` and writes regenerated products under `outputs/`.


In [ ]:
def equalize_lists(data):
    # Find the maximum length among the lists
    max_length = max(len(lst) for lst in data.values())

    # Equalize lengths by appending np.nan to shorter lists
    equalized_data = {
        key: lst + [np.nan] * (max_length - len(lst)) if len(lst) < max_length else lst
        for key, lst in data.items()
    }

    return equalized_data

### Archived upstream-dependent cell

This original code cell depended on local upstream files or intermediate products that are intentionally not included in this time-series-first supplement. The supported workflow starts from `data/time-series/` and writes regenerated products under `outputs/`.


### Archived upstream-dependent cell

This original code cell depended on local upstream files or intermediate products that are intentionally not included in this time-series-first supplement. The supported workflow starts from `data/time-series/` and writes regenerated products under `outputs/`.


### Archived upstream-dependent cell

This original code cell depended on local upstream files or intermediate products that are intentionally not included in this time-series-first supplement. The supported workflow starts from `data/time-series/` and writes regenerated products under `outputs/`.


## Sorbitol

In [ ]:
parentDir = str(legacy_speed_tree(assay="Sorbitol"))
outputDir = str(ROOT / "outputs" / "bead" / "sorbitol")
os.chdir(parentDir)

# Initialize the list to store time, data, and sucrose concentrations
all_time = []
all_data = []
concentrations = []

tau_values = {}
t0_values = {}
A_values = {}
tau_inc = {}

# Desired length for all data

target_length = 118000

# Walk through all directories and subdirectories in parentDir
for root, dirs, files in os.walk(parentDir):
    print(root, dirs, files)
    # Filter to only get CSV files in the current directory
    csv_files = [f for f in files if f.endswith('.csv')]

    # If the directory contains CSV files, process them
    if csv_files:
        tau_values[str(root[-5:])] = []
        t0_values[str(root[-5:])] = []
        A_values[str(root[-5:])] = []
        tau_inc[str(root[-5:])] = []
        # Get the time data from the first CSV file to create the time axis
        time = np.array(pd.read_csv(os.path.join(root, csv_files[2]))['Unnamed: 0'][:target_length]) / 300
       
        # Initialize an array to hold data from all CSV files in this folder
        allcells = np.zeros((len(csv_files), 3800))
        
        # Loop through the CSV files in this directory
        for count, fileName in enumerate(csv_files):
            file_path = os.path.join(root, fileName)
            mat_data = pd.read_csv(file_path)
            if root[-5:-2] == "300":
                print(file_path)

            # If the data is shorter than the target length, pad it with zeros
            data = np.array(mat_data['0'])[:target_length]  # Trim data if it's longer

            if len(data) < target_length:
                # Pad with zeros if shorter
                data = np.pad(data, (0, target_length - len(data)), mode='constant', constant_values=np.nan)
            
            # Apply median filtering
            data = scipy.signal.medfilt(data, kernel_size=median_kernel_size(time, window_s=1.0)) # 1 s median filter

            # NOTE FROM NAVISH: MEDFILT MAY INTRODUCE EDGE EFFECTS. WE MIGHT WANT TO USE scipy.ndimage.median_filter INSTEAD.

            new_time, interp_data = bin_data(time, data)

            tau_dec, t0_dec, A_dec, tau_inc_v = calculate_time_constants(new_time, interp_data, str(root[-5:-2]))
            if ((tau_dec > 0) & (tau_dec < 40)) and (A_dec<1.6):
                tau_values[str(root[-5:])] += [tau_dec]
                t0_values[str(root[-5:])] += [t0_dec]
                A_values[str(root[-5:])] += [A_dec]
                tau_inc[str(root[-5:])] += [tau_inc_v]  # Placeholder for tau_inc, not calculated here
            
            allcells[count] = interp_data

        # Calculate the mean and standard deviation for this folder's data
        data_mean = np.average(allcells, axis=0)
        
        time_mult = 10
        data_mean_norm = data_mean / np.average(data_mean[:180 * time_mult])  # Normalize by the first part of the data  #normally 300 for Luis's data

        data_std = np.std(allcells, axis=0) / np.average(data_mean[:180 * time_mult])  #normally 300 for Luis's data

        # Calculate the upper and lower bounds for the shaded area
        upper = data_mean_norm + 0.5 * data_std
        lower = data_mean_norm - 0.5 * data_std

        # Append time, normalized data, and sucrose concentration
        all_time.append(new_time)
        all_data.append(data_mean_norm)

        # Extract sucrose concentration from the directory name
        sucrose_concentration = os.path.basename(root)[:-2]  # Assuming sucrose concentration is in the last 3 characters
        concentrations.append(float(sucrose_concentration))  # Convert to float for sorting

        # Save the data to a CSV file for this sucrose concentration
        sucrose_output_dir = os.path.join(outputDir, f"{sucrose_concentration}mM")
        #os.makedirs(sucrose_output_dir, exist_ok=True)  # Create directory if it doesn't exist

        # Save time and normalized data to CSV
        df_output = pd.DataFrame({
            'Time': new_time,
            'Normalized Data': data_mean_norm
        })
        output_file = os.path.join(sucrose_output_dir, f"normalized_data_{sucrose_concentration}mM.csv")
        #df_output.to_csv(output_file, index=False)

# Sort the concentrations and associated data
sorted_indices = np.argsort(concentrations)
sorted_concentrations = [concentrations[i] for i in sorted_indices]
sorted_data = [all_data[i] for i in sorted_indices]
sorted_times = [all_time[i] for i in sorted_indices]         

In [ ]:
def equalize_lists(data):
    # Find the maximum length among the lists
    max_length = max(len(lst) for lst in data.values())

    # Equalize lengths by appending np.nan to shorter lists
    equalized_data = {
        key: lst + [np.nan] * (max_length - len(lst)) if len(lst) < max_length else lst
        for key, lst in data.items()
    }

    return equalized_data

### Archived upstream-dependent cell

This original code cell depended on local upstream files or intermediate products that are intentionally not included in this time-series-first supplement. The supported workflow starts from `data/time-series/` and writes regenerated products under `outputs/`.


## Plot Concentrations of Averages 

### Archived upstream-dependent cell

This original code cell depended on local upstream files or intermediate products that are intentionally not included in this time-series-first supplement. The supported workflow starts from `data/time-series/` and writes regenerated products under `outputs/`.


## Plot Tau Decrease Values

### Archived upstream-dependent cell

This original code cell depended on local upstream files or intermediate products that are intentionally not included in this time-series-first supplement. The supported workflow starts from `data/time-series/` and writes regenerated products under `outputs/`.


### Archived upstream-dependent cell

This original code cell depended on local upstream files or intermediate products that are intentionally not included in this time-series-first supplement. The supported workflow starts from `data/time-series/` and writes regenerated products under `outputs/`.


## Sodium

In [ ]:
parentDir = str(legacy_speed_tree(assay="Sodium"))
outputDir = str(ROOT / "outputs" / "bead" / "sodium")
os.chdir(parentDir)

# Initialize the list to store time, data, and sucrose concentrations
all_time = []
all_data = []
concentrations = []

tau_values = {}
t0_values = {}
A_values = {}
tau_inc = {}

# Desired length for all data

target_length = 118000

# Walk through all directories and subdirectories in parentDir
for root, dirs, files in os.walk(parentDir):
    print(root, dirs, files)
    # Filter to only get CSV files in the current directory
    csv_files = [f for f in files if f.endswith('.csv')]

    # If the directory contains CSV files, process them
    if csv_files:
        tau_values[str(root[-5:])] = []
        t0_values[str(root[-5:])] = []
        A_values[str(root[-5:])] = []
        tau_inc[str(root[-5:])] = []
        # Get the time data from the first CSV file to create the time axis
        time = np.array(pd.read_csv(os.path.join(root, csv_files[2]))['Unnamed: 0'][:target_length]) / 300
       
        # Initialize an array to hold data from all CSV files in this folder
        allcells = np.zeros((len(csv_files), 3800))
        
        # Loop through the CSV files in this directory
        for count, fileName in enumerate(csv_files):
            file_path = os.path.join(root, fileName)
            mat_data = pd.read_csv(file_path)
            if root[-5:-2] == "300":
                print(file_path)

            # If the data is shorter than the target length, pad it with zeros
            data = np.array(mat_data['0'])[:target_length]  # Trim data if it's longer

            if len(data) < target_length:
                # Pad with zeros if shorter
                data = np.pad(data, (0, target_length - len(data)), mode='constant', constant_values=np.nan)
            
            # Apply median filtering
            data = scipy.signal.medfilt(data, kernel_size=median_kernel_size(time, window_s=1.0)) # 1 s median filter

            new_time, interp_data = bin_data(time, data)

            tau_dec, t0_dec, A_dec, tau_inc_v = calculate_time_constants(new_time, interp_data, str(root[-5:-2]))
            if ((tau_dec > 0) & (tau_dec < 40)) and (A_dec<1.6):
                tau_values[str(root[-5:])] += [tau_dec]
                t0_values[str(root[-5:])] += [t0_dec]
                A_values[str(root[-5:])] += [A_dec]
                tau_inc[str(root[-5:])] += [tau_inc_v]  # Placeholder for tau_inc, not calculated here
            
            allcells[count] = interp_data

        # Calculate the mean and standard deviation for this folder's data
        data_mean = np.average(allcells, axis=0)
        
        time_mult = 10
        data_mean_norm = data_mean / np.average(data_mean[:180 * time_mult])  # Normalize by the first part of the data  #normally 300 for Luis's data

        data_std = np.std(allcells, axis=0) / np.average(data_mean[:180 * time_mult])  #normally 300 for Luis's data

        # Calculate the upper and lower bounds for the shaded area
        upper = data_mean_norm + 0.5 * data_std
        lower = data_mean_norm - 0.5 * data_std

        # Append time, normalized data, and sucrose concentration
        all_time.append(new_time)
        all_data.append(data_mean_norm)

        # Extract sucrose concentration from the directory name
        sucrose_concentration = os.path.basename(root)[:-2]  # Assuming sucrose concentration is in the last 3 characters
        concentrations.append(float(sucrose_concentration))  # Convert to float for sorting

        # Save the data to a CSV file for this sucrose concentration
        sucrose_output_dir = os.path.join(outputDir, f"{sucrose_concentration}mM")
        #os.makedirs(sucrose_output_dir, exist_ok=True)  # Create directory if it doesn't exist

        # Save time and normalized data to CSV
        df_output = pd.DataFrame({
            'Time': new_time,
            'Normalized Data': data_mean_norm
        })
        output_file = os.path.join(sucrose_output_dir, f"normalized_data_{sucrose_concentration}mM.csv")
        #df_output.to_csv(output_file, index=False)

# Sort the concentrations and associated data
sorted_indices = np.argsort(concentrations)
sorted_concentrations = [concentrations[i] for i in sorted_indices]
sorted_data = [all_data[i] for i in sorted_indices]
sorted_times = [all_time[i] for i in sorted_indices]

In [ ]:
def equalize_lists(data):
    # Find the maximum length among the lists
    max_length = max(len(lst) for lst in data.values())

    # Equalize lengths by appending np.nan to shorter lists
    equalized_data = {
        key: lst + [np.nan] * (max_length - len(lst)) if len(lst) < max_length else lst
        for key, lst in data.items()
    }

    return equalized_data

### Archived upstream-dependent cell

This original code cell depended on local upstream files or intermediate products that are intentionally not included in this time-series-first supplement. The supported workflow starts from `data/time-series/` and writes regenerated products under `outputs/`.


### Archived upstream-dependent cell

This original code cell depended on local upstream files or intermediate products that are intentionally not included in this time-series-first supplement. The supported workflow starts from `data/time-series/` and writes regenerated products under `outputs/`.


### Archived upstream-dependent cell

This original code cell depended on local upstream files or intermediate products that are intentionally not included in this time-series-first supplement. The supported workflow starts from `data/time-series/` and writes regenerated products under `outputs/`.


### Archived upstream-dependent cell

This original code cell depended on local upstream files or intermediate products that are intentionally not included in this time-series-first supplement. The supported workflow starts from `data/time-series/` and writes regenerated products under `outputs/`.


## Clockwise

In [ ]:
parentDir = str(legacy_speed_tree(assay="Clockwise"))
outputDir = str(ROOT / "outputs" / "bead" / "clockwise")

# Initialize the list to store time, data, and sucrose concentrations
all_time = []
all_data = []
concentrations = []

tau_values = {}
t0_values = {}
A_values = {}
tau_inc = {}

# Desired length for all data

target_length = 118000

# Walk through all directories and subdirectories in parentDir
for root, dirs, files in os.walk(parentDir):
    print(root, dirs, files)
    # Filter to only get CSV files in the current directory
    csv_files = [f for f in files if f.endswith('.csv')]

    # If the directory contains CSV files, process them
    if csv_files:
        tau_values[str(root[-5:])] = []
        t0_values[str(root[-5:])] = []
        A_values[str(root[-5:])] = []
        tau_inc[str(root[-5:])] = []
        # Get the time data from the first CSV file to create the time axis
        time = np.array(pd.read_csv(os.path.join(root, csv_files[2]))['Unnamed: 0'][:target_length]) / 300
       
        # Initialize an array to hold data from all CSV files in this folder
        allcells = np.zeros((len(csv_files), 3800))
        
        # Loop through the CSV files in this directory
        for count, fileName in enumerate(csv_files):
            file_path = os.path.join(root, fileName)
            mat_data = pd.read_csv(file_path)
            if root[-5:-2] == "300":
                print(file_path)

            # If the data is shorter than the target length, pad it with zeros
            data = np.array(mat_data['0'])[:target_length]  # Trim data if it's longer

            if len(data) < target_length:
                # Pad with zeros if shorter
                data = np.pad(data, (0, target_length - len(data)), mode='constant', constant_values=np.nan)
            
            # Apply median filtering
            data = scipy.signal.medfilt(data, kernel_size=median_kernel_size(time, window_s=1.0)) # 1 s median filter

            new_time, interp_data = bin_data(time, data)

            tau_dec, t0_dec, A_dec, tau_inc_v = calculate_time_constants(new_time, interp_data, str(root[-5:-2]))
            if ((tau_dec > 0) & (tau_dec < 40)) and (A_dec<1.6):
                tau_values[str(root[-5:])] += [tau_dec]
                t0_values[str(root[-5:])] += [t0_dec]
                A_values[str(root[-5:])] += [A_dec]
                tau_inc[str(root[-5:])] += [tau_inc_v]  # Placeholder for tau_inc, not calculated here
            
            allcells[count] = interp_data

        # Calculate the mean and standard deviation for this folder's data
        data_mean = np.average(allcells, axis=0)
        
        time_mult = 10
        data_mean_norm = data_mean / np.average(data_mean[:180 * time_mult])  # Normalize by the first part of the data  #normally 300 for Luis's data

        data_std = np.std(allcells, axis=0) / np.average(data_mean[:180 * time_mult])  #normally 300 for Luis's data

        # Calculate the upper and lower bounds for the shaded area
        upper = data_mean_norm + 0.5 * data_std
        lower = data_mean_norm - 0.5 * data_std

        # Append time, normalized data, and sucrose concentration
        all_time.append(new_time)
        all_data.append(data_mean_norm)

        # Extract sucrose concentration from the directory name
        sucrose_concentration = os.path.basename(root)[:-2]  # Assuming sucrose concentration is in the last 3 characters
        concentrations.append(float(sucrose_concentration))  # Convert to float for sorting

        # # Save the data to a CSV file for this sucrose concentration
        # sucrose_output_dir = os.path.join(outputDir, f"{sucrose_concentration}mM")
        # #os.makedirs(sucrose_output_dir, exist_ok=True)  # Create directory if it doesn't exist

        # # Save time and normalized data to CSV
        # df_output = pd.DataFrame({
        #     'Time': new_time,
        #     'Normalized Data': data_mean_norm
        # })
        # output_file = os.path.join(sucrose_output_dir, f"normalized_data_{sucrose_concentration}mM.csv")
        # #df_output.to_csv(output_file, index=False)

# Sort the concentrations and associated data
sorted_indices = np.argsort(concentrations)
sorted_concentrations = [concentrations[i] for i in sorted_indices]
sorted_data = [all_data[i] for i in sorted_indices]
sorted_times = [all_time[i] for i in sorted_indices]

In [ ]:
def equalize_lists(data):
    # Find the maximum length among the lists
    max_length = max(len(lst) for lst in data.values())

    # Equalize lengths by appending np.nan to shorter lists
    equalized_data = {
        key: lst + [np.nan] * (max_length - len(lst)) if len(lst) < max_length else lst
        for key, lst in data.items()
    }

    return equalized_data

### Archived upstream-dependent cell

This original code cell depended on local upstream files or intermediate products that are intentionally not included in this time-series-first supplement. The supported workflow starts from `data/time-series/` and writes regenerated products under `outputs/`.


### Archived upstream-dependent cell

This original code cell depended on local upstream files or intermediate products that are intentionally not included in this time-series-first supplement. The supported workflow starts from `data/time-series/` and writes regenerated products under `outputs/`.


### Archived upstream-dependent cell

This original code cell depended on local upstream files or intermediate products that are intentionally not included in this time-series-first supplement. The supported workflow starts from `data/time-series/` and writes regenerated products under `outputs/`.


### Archived upstream-dependent cell

This original code cell depended on local upstream files or intermediate products that are intentionally not included in this time-series-first supplement. The supported workflow starts from `data/time-series/` and writes regenerated products under `outputs/`.
